In [39]:
import sys
sys.path.append('.')

from src.data_utils import load_data, prepare_dataset, split_data
from src.dataset import NextTokenDataset
import torch
from torch.utils.data import DataLoader

print("Все модули успешно импортированы!")

Все модули успешно импортированы!


In [40]:

df_raw = load_data('data/training.1600000.processed.noemoticon.csv', n_rows=50000)

df_clean = prepare_dataset(df_raw)

# Смотрим на результат
print("\nПримеры очищенных текстов:")
for i in range(3):
    print(f"  {i+1}. {df_clean['clean_text'].iloc[i]}")
    print(f"     Токены: {df_clean['tokens'].iloc[i]}")
    print()

Загрузка данных из data/training.1600000.processed.noemoticon.csv...
Загружено 50000 строк
Очистка текстов...
Токенизация...
Готово! 49915 примеров

Примеры очищенных текстов:
  1. a thats a bummer. you shoulda got david carr of third day to do it. d
     Токены: ['a', 'thats', 'a', 'bummer.', 'you', 'shoulda', 'got', 'david', 'carr', 'of', 'third', 'day', 'to', 'do', 'it.', 'd']

  2. is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!
     Токены: ['is', 'upset', 'that', 'he', 'cant', 'update', 'his', 'facebook', 'by', 'texting', 'it...', 'and', 'might', 'cry', 'as', 'a', 'result', 'school', 'today', 'also.', 'blah!']

  3. i dived many times for the ball. managed to save 50 the rest go out of bounds
     Токены: ['i', 'dived', 'many', 'times', 'for', 'the', 'ball.', 'managed', 'to', 'save', '50', 'the', 'rest', 'go', 'out', 'of', 'bounds']



In [41]:
# solution.ipynb
# Ячейка 3: Разбиваем на train/val/test и сохраняем

train, val, test = split_data(df_clean)

print("\nПроверяем сохраненные файлы:")
!ls -la data/

Train: 39932 примеров
Val: 4991 примеров
Test: 4992 примеров

Проверяем сохраненные файлы:
total 491064
drwxr-xr-x@  6 tochi  staff        192 Mar 15 14:13 .
drwxr-xr-x@ 14 tochi  staff        448 Mar 15 13:54 ..
-rw-r--r--@  1 tochi  staff    1269790 Mar 15 14:13 test.csv
-rw-r--r--@  1 tochi  staff   10068549 Mar 15 14:13 train.csv
-rw-rw-r--@  1 tochi  staff  238803811 Mar 15 13:53 training.1600000.processed.noemoticon.csv
-rw-r--r--@  1 tochi  staff    1270349 Mar 15 14:13 val.csv


In [42]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    inputs, targets = zip(*batch)

    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    
    return inputs_padded, targets_padded

train_dataset = NextTokenDataset('data/train.csv', max_len=20)
val_dataset = NextTokenDataset('data/val.csv', max_len=20)
test_dataset = NextTokenDataset('data/test.csv', max_len=20)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    collate_fn=collate_batch,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=collate_batch,
    num_workers=0
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=collate_batch,
    num_workers=0
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Размер словаря: 50790
Всего примеров после фильтрации: 38751
Размер словаря: 12096
Всего примеров после фильтрации: 4832
Размер словаря: 12178
Всего примеров после фильтрации: 4856

Train batches: 1211
Val batches: 151
Test batches: 152


In [43]:
x, y = next(iter(train_loader))
print(f"Форма входных данных: {x.shape}") 
print(f"Форма целевых данных: {y.shape}") 
print(f"Размер словаря: {train_dataset.vocab_size}")
print(f"Индекс паддинга: {train_dataset.pad_idx}")

print(f"\nПервый пример входа: {x[0][:10]}...")
print(f"Первый пример цели: {y[0][:10]}...")

Форма входных данных: torch.Size([32, 19])
Форма целевых данных: torch.Size([32, 19])
Размер словаря: 50790
Индекс паддинга: 0

Первый пример входа: tensor([   36,   513,  1046,    40,    97, 11614,    78,  1006, 44378,   173])...
Первый пример цели: tensor([  513,  1046,    40,    97, 11614,    78,  1006, 44378,   173,   317])...


In [44]:
from src.lstm_model import LSTMAutocomplete

vocab_size = train_dataset.vocab_size
pad_idx = train_dataset.pad_idx

model = LSTMAutocomplete(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    pad_idx=pad_idx
)

print(model)
print(f"\nВсего параметров: {sum(p.numel() for p in model.parameters())}")

LSTMAutocomplete(
  (embedding): Embedding(50790, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=50790, bias=True)
)

Всего параметров: 20475750


In [45]:
x, y = next(iter(train_loader))

with torch.no_grad():
    outputs = model(x)
    
print(f"Вход: {x.shape}")
print(f"Выход: {outputs.shape}")  # (batch_size, seq_len, vocab_size)
print(f"Логиты для первого токена первого примера: {outputs[0, 0, :5]}...")

Вход: torch.Size([32, 19])
Выход: torch.Size([32, 19, 50790])
Логиты для первого токена первого примера: tensor([-0.0394, -0.0428, -0.0173, -0.0375,  0.0530])...


In [46]:
sample_idx = 0
sample_tokens = train_dataset.df.iloc[sample_idx]['token_list']
print(f"Полный текст: {' '.join(sample_tokens)}")

start_tokens = train_dataset.text_to_ids(sample_tokens[:3])
print(f"Начало (3 токена): {train_dataset.ids_to_text(start_tokens)}")

generated_ids = model.generate(start_tokens, max_new_tokens=5)
generated_text = train_dataset.ids_to_text(generated_ids)
print(f"Сгенерировано: {generated_text}")

Полный текст: feels alone because he has no friends on twitter et miley cyrus doesnt answer him.
Начало (3 токена): feels alone because
Сгенерировано: feels alone because beer, beer, swfoocamp up up


In [47]:
!zip -r data.zip data/ src/
print("Данные подготовлены!")

updating: data/ (stored 0%)
updating: data/val.csv (deflated 68%)
updating: data/test.csv (deflated 68%)
updating: data/training.1600000.processed.noemoticon.csv (deflated 64%)
updating: data/train.csv (deflated 68%)
updating: src/ (stored 0%)
updating: src/dataset.py (deflated 64%)
updating: src/data_utils.py (deflated 56%)
updating: src/evaluate.py (deflated 70%)
updating: src/lstm_model.py (deflated 71%)
updating: src/train_lstm.py (deflated 64%)
Данные подготовлены!
